<a href="https://www.kaggle.com/code/criser2013/univariate-feature-engineering?scriptVersionId=350621912" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Univariate feature engineering techniques
## Mutual information
As the name suggests, it calculates a "relationship score" between every feature and target, can detect any kind of relationship between 2 quantities. Works like a way for measuring how much information can we get/lose if a variable reduces or increases. The formal definition is similar to information gain and entropy on decision trees. 
Mutual information scores works like $r$ coefficient in linear regression. Score range goes from $0$ to $\inf$. When MI score is 0 means that variables are independent. The larger MI score is, variables are more related. Throughout an upper bound is not defined, usually values above 2 are uncommon. Also mutual information can be used in quantitative and qualitative variables.


**Considerations:**
- If continuous variables are discretized, results doesn't reflects true results for every value. MI score will depend on the strategy used for building bins.
- Assumes independence in the relationship between the target and the chosen feature. So if many single features receives lower MI scores, doesn't mean that should be removed.
- Biased when there are many unique values in  some variables (thinking that many unique values means a stronger information source).
- Assumes a monotonic relationship, meaning that as one variable increases the other variable also increases or decreases.
- Computationally expensive on big or highly dimensional datasets.

In [ ]:
from sklearn.model_selection import train_test_split
from pandas import read_csv
import matplotlib.pyplot as plt

DATA = read_csv("/kaggle/input/datasets/debayank2024/house-price-prediction/modified_data.csv")
NUMS = ["sqft_living", "bathrooms", "bedrooms", "sqft_lot", "floors", "sqft_above", "sqft_basement", "yr_built", "price_per_sqft"]
CATS = ["waterfront", "view", "condition"]
TARGET = "price"

data = DATA.drop(columns=["street", "statezip"])

x_train, x_test, y_train, y_test = train_test_split(data[NUMS + CATS + ["city"]], data[TARGET], test_size=0.2, random_state=123)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

def eval_feature_selection(x_train, x_test, y_train, y_test, feature_selection_func, kbest= False, k = 5):
    TRANSFORMER = ColumnTransformer([
        ("standarized", StandardScaler(), NUMS),
        ("encoding", OrdinalEncoder(), ["city"]),
        ("pass", "passthrough", CATS)
    ], verbose_feature_names_out=False)
    
    TRANSFORMER.set_output(transform="pandas")

    KBEST = SelectKBest(feature_selection_func, k=k) if kbest else feature_selection_func

    PIPELINE = Pipeline(steps=[
        ("processing", TRANSFORMER),
        ("kbest", KBEST)
    ])

    X_train_kbest = PIPELINE.fit_transform(x_train, y_train)
    X_test_kbest = PIPELINE.transform(x_test)

    X_train = TRANSFORMER.transform(x_train)
    X_test = TRANSFORMER.transform(x_test)

    SIMPLE_MODEL = GradientBoostingRegressor(random_state=123, learning_rate=0.05, n_estimators=200)
    SIMPLE_MODEL.fit(X_train, y_train)
    
    SIMPLE_PREDS = SIMPLE_MODEL.predict(X_test)
    SIMPLE_SQUARED_ERROR = mean_squared_error(y_test, SIMPLE_PREDS)
    
    KBEST_MODEL = GradientBoostingRegressor(random_state=123, learning_rate=0.05, n_estimators=200)
    KBEST_MODEL.fit(X_train_kbest, y_train)
    
    KBEST_PREDS = KBEST_MODEL.predict(X_test_kbest)
    KBEST_SQUARED_ERROR = mean_squared_error(y_test, KBEST_PREDS)
    
    print("Squared error on simple dataset: ", SIMPLE_SQUARED_ERROR)
    print("Squared error with feature selection: ", KBEST_SQUARED_ERROR)

    return KBEST

In [ ]:
from sklearn.feature_selection import mutual_info_regression

KBEST = eval_feature_selection(x_train, x_test, y_train, y_test, mutual_info_regression, True, 5)
FEATURES = KBEST.feature_names_in_
SCORES = KBEST.scores_

fig, ax = plt.subplots()

ax.barh(FEATURES, SCORES, align='center')
ax.yaxis.set_inverted(True)  # arrange data from top to bottom
ax.set_xlabel('Feature name')
ax.set_title('Mutual Information score')

print("Selected features: ", KBEST.get_feature_names_out().tolist())

## ANOVA F-test
Works as the same way as MI coefficient,is a statistical method that finds how ell a feature distinguishes between different classes by comparing variability between classes to the variability within each class. So it calculates F-statistic for each feature with respect to the target variable. Higher values shows a great correlation with the target variable.

**Considerations:**
- Assumes normality among data in each class.
- Sensitive to outliners.
- May not perform well when dataset contains highly correlated features.
- Ignores feature interactions.
- Works for classification and regression tasks, but is highly recommended for classification.

In [ ]:
from sklearn.feature_selection import f_regression

KBEST = eval_feature_selection(x_train, x_test, y_train, y_test, f_regression, True, 5)
FEATURES = KBEST.feature_names_in_
SCORES = KBEST.scores_

fig, ax = plt.subplots()

ax.barh(FEATURES, SCORES, align='center')
ax.yaxis.set_inverted(True)  # arrange data from top to bottom
ax.set_xlabel('Feature name')
ax.set_title('F-score')

print("Selected features: ", KBEST.get_feature_names_out().tolist())

## R-score
As the previous, works for scoring correlation among features and a target variable. In contrast to the others, works just for regression tasks with continuous variables. Is the `r` coefficient on linear regression, so range goes from $-1$ to $1$ bound values tell us that both variables are highly correlated. Negative values means that they are inversevely correlated (when one increases the other decreases and viceverse) and positive values means the opposite. A `0` means that there is not relationship between variables.

**Considerations:**
- Ignores relationships between variables.
- Only works for continuous features.
- Assumes a linear trend among feature and target variable.
- Sensitive to outliners.
- Assumes normal distribution.
- Not recommended for sparse data.

## Chi-squared test
As the name suggests this approach consists on performing an independence chi-squared test between features and target variable. However, limitation of chi-squared test are present, so just works for classification tasks over qualitative variables.

**Considerations:**
- Only works for classification tasks.
- Just performs on qualitative variables, quantitative variables must be discretized.
- Ignores relationships among variables

## Variance threshold
This technique goes far from the other methods, because consists on calcutating the variance of each feature and removes those with variance lower than a selected threshold. Bases on the fact that low variance variables contains little information because almost constant across samples, works fine for removing noise.

**Considerations:**
- Doesn't considers target variable.
- Is computationally efficient.
- Can't detect correlated features.
- The assumption could be false in some cases.
- Check variables distribution before choosing a threshold.

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# By default it removes variables with 0 variance
KBEST = eval_feature_selection(x_train, x_test, y_train, y_test, VarianceThreshold(), False)

FEATURES = KBEST.feature_names_in_
SCORES = KBEST.variances_

fig, ax = plt.subplots()

ax.barh(FEATURES, SCORES, align='center')
ax.yaxis.set_inverted(True)  # arrange data from top to bottom
ax.set_xlabel('Feature name')
ax.set_title('Variances')

print("Selected features: ", KBEST.get_feature_names_out().tolist())

We have seen most used approaches to select features based on univariate techniques. Most bases on statical tests and assumptions often `KBest` **Scikit** util is used, but there are other strategies to use this techniques:

- **`KBest`:** Selects the `k` highest scored features.
- **`SelectPercentile`:** Performs score function and holds the desired features percentalice to keep.
- **`SelectFpr`:** Performs a false positive rate test, calculates p-values and just holds those below specified $\alpha$ (significance coefficient). **Only works on classification tasks**.
- **`SelectFdr`:** Uses Benjamini-Hochberg procedure, calculates p-values for an estimated discovery rate and holds only those corresponding false discovery rate.
- **`SelectFwe`:** Similar to the previous but calculates Family-wise error rate.

Most often, those strategies are enough for removing noisy features that could obscure model predictions, however, the biggest weakness is that they don't consider features relationships.